# EXP_004 — Frozen REVE + Leadfield B_sim: Seizure Detection (CHB-MIT Cross-Patient)

Ports the EXP_003 result (B_sim injection into frozen REVE, +14.8% on motor imagery)
to **seizure detection** on CHB-MIT. Answers: *does the physics gain generalise to
a different task and pathology?*

**Setup:** same three conditions as EXP_003 —
- A — frozen REVE baseline
- B — frozen REVE + α·B_sim channel mixing
- C — B + electrode jitter 5 mm + SVD noise 10 %

**Evaluation:** cross-patient leave-one-patient-out (LOPO). Primary metric: AUC.
Secondary: sensitivity (seizure recall), specificity.

**Data:** CHB-MIT (PhysioNet). Auto-downloaded — no local data needed.
Patients chb01–chb05 (5 patients, 7 seizures each on average, 256 Hz → 200 Hz).

In [ ]:
import subprocess, sys, os

IN_COLAB  = 'google.colab'  in sys.modules
IN_KAGGLE = 'kaggle_secrets' in sys.modules or os.path.exists('/kaggle')

if IN_COLAB or IN_KAGGLE:
    repo = '/content/PhysREVE' if IN_COLAB else '/kaggle/working/PhysREVE'
    if not os.path.exists(repo):
        subprocess.check_call(['git', 'clone', '-q',
            'https://github.com/UgoBruzadin/PhysREVE.git', repo])
    else:
        subprocess.check_call(['git', '-C', repo, 'pull', '-q'])
    if repo not in sys.path:
        sys.path.insert(0, repo)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'mne>=1.6', 'transformers', 'huggingface_hub', 'einops',
        'requests', 'scipy'])
    print('Environment ready.')
else:
    print('Local environment.')

import torch
print(f'PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 1. Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
import matplotlib.pyplot as plt
import warnings, json
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Device: {DEVICE}')


## 2. Download CHB-MIT (patients chb01–chb05)

In [ ]:
from physreve.datasets.chbmit import download_patient, parse_summary, load_patient_epochs

DATA_DIR = './data/chb-mit'

# Files with confirmed seizures for chb01-chb05
# (subset kept small so LOPO runs in <2h on a T4 GPU)
PATIENT_FILES = {
    'chb01': ['chb01_03.edf', 'chb01_04.edf', 'chb01_15.edf',
              'chb01_16.edf', 'chb01_18.edf', 'chb01_21.edf', 'chb01_26.edf',
              'chb01_01.edf', 'chb01_02.edf'],   # last two: interictal pool
    'chb02': ['chb02_16.edf', 'chb02_17.edf', 'chb02_19.edf',
              'chb02_01.edf', 'chb02_02.edf'],
    'chb03': ['chb03_01.edf', 'chb03_02.edf', 'chb03_03.edf',
              'chb03_04.edf', 'chb03_34.edf', 'chb03_35.edf', 'chb03_36.edf'],
    'chb04': ['chb04_05.edf', 'chb04_08.edf', 'chb04_28.edf',
              'chb04_01.edf', 'chb04_02.edf'],
    'chb05': ['chb05_06.edf', 'chb05_13.edf', 'chb05_16.edf',
              'chb05_17.edf', 'chb05_22.edf',
              'chb05_01.edf', 'chb05_02.edf'],
}

PATIENTS = list(PATIENT_FILES.keys())

for pid, files in PATIENT_FILES.items():
    print(f'Downloading {pid}...')
    download_patient(pid, DATA_DIR, files)
    print(f'  {pid} ready')


## 3. Extract windows per patient

In [ ]:
from pathlib import Path

patient_data = {}   # pid -> {'X': (N,C,T), 'y': (N,), 'ch_names': [...]}

WINDOW_SEC   = 4.0
STRIDE_SEC   = 2.0
TARGET_SFREQ = 200.0

for pid in PATIENTS:
    summary_path = Path(DATA_DIR) / pid / f'{pid}-summary.txt'
    seizure_info = parse_summary(str(summary_path))

    ictal_X, interictal_X, ch_names = load_patient_epochs(
        data_dir=DATA_DIR, patient_id=pid, seizure_info=seizure_info,
        window_sec=WINDOW_SEC, stride_sec=STRIDE_SEC,
        target_sfreq=TARGET_SFREQ,
        ictal_buffer_sec=2.0, interictal_gap_sec=60.0,
        max_interictal_per_file=200, verbose=False,
    )

    X   = np.concatenate([ictal_X, interictal_X], axis=0)
    y   = np.array([1]*len(ictal_X) + [0]*len(interictal_X), dtype=np.int64)
    patient_data[pid] = {'X': X, 'y': y, 'ch_names': ch_names}
    print(f'{pid}: ictal={len(ictal_X)}  interictal={len(interictal_X)}  '
          f'ch={len(ch_names)}  T={X.shape[-1]}')

# All patients must share the same channel set
CH_NAMES = patient_data[PATIENTS[0]]['ch_names']
N_CH     = len(CH_NAMES)
T_WIN    = patient_data[PATIENTS[0]]['X'].shape[-1]
print(f'\nChannels ({N_CH}): {CH_NAMES}')
print(f'Window  : {T_WIN} samples = {T_WIN/TARGET_SFREQ:.1f} s @ {TARGET_SFREQ:.0f} Hz')


## 4. Build bipolar leadfield + electrode positions for REVE

In [ ]:
from physreve.datasets.chbmit import build_leadfield_bipolar

print(f'Building bipolar leadfield for {N_CH} channels...')
L_col_np, L_row_np, src_pos_np, info_ref, elec_xyz_np, valid_mask = \
    build_leadfield_bipolar(CH_NAMES, sfreq=TARGET_SFREQ, grid_pos=10.0, verbose=True)

N_SRC = L_col_np.shape[1]
print(f'Leadfield: {L_col_np.shape}  |  B_sim will be ({N_CH},{N_CH})')

L_row_t  = torch.tensor(L_row_np,   dtype=torch.float32).to(DEVICE)
B_sim    = (L_row_t @ L_row_t.T)                                  # (C, C)
POSITIONS = torch.tensor(elec_xyz_np, dtype=torch.float32).to(DEVICE)  # (C, 3)
# elec_xyz_np contains midpoint positions of bipolar pairs — used directly as REVE positions
print(f'B_sim range: [{B_sim.min():.2f}, {B_sim.max():.2f}]')
print(f'Positions : {POSITIONS.shape}')


## 5. Load frozen REVE

> Requires HuggingFace login (free account). Run `huggingface-cli login` or use `notebook_login()` below.

In [ ]:
from huggingface_hub import notebook_login
from transformers import AutoModel

notebook_login()   # paste your HF read token when prompted

print('Loading REVE-Base...')
reve_model = AutoModel.from_pretrained('brain-bzh/reve-base', trust_remote_code=True)
reve_model = reve_model.to(DEVICE).eval()
for p in reve_model.parameters():
    p.requires_grad = False

n_params = sum(p.numel() for p in reve_model.parameters())
print(f'REVE frozen: {n_params:,} parameters')

# Probe output shape
with torch.no_grad():
    _x = torch.randn(2, N_CH, T_WIN).to(DEVICE)
    _p = POSITIONS.unsqueeze(0).expand(2, -1, -1)
    _o = reve_model(_x, _p)
    _f = _o.last_hidden_state if hasattr(_o, 'last_hidden_state') else (_o[0] if isinstance(_o,(tuple,list)) else _o)
print(f'REVE output shape: {_f.shape}')
REVE_FEAT_DIM = _f.mean(dim=list(range(1, _f.dim()-1))).shape[-1]
print(f'Feature dim: {REVE_FEAT_DIM}')
del _x, _p, _o, _f


## 6. Model: frozen REVE + learnable B_sim channel mixing

In [ ]:
class LeadfieldREVEExtractor(nn.Module):
    """Frozen REVE + learnable leadfield channel-mixing.
    Conditions:
      sigma_elec=0, sigma_svd=0   -> B  (pure bias)
      sigma_elec=0.005, sigma_svd=0.10 -> C  (bias + jitter)
    """
    def __init__(self, reve_model, positions, B_sim,
                 sigma_elec=0.0, sigma_svd=0.0):
        super().__init__()
        self.reve       = reve_model
        self.sigma_elec = sigma_elec
        self.sigma_svd  = sigma_svd
        self.register_buffer('positions', positions)
        self.register_buffer('B_sim',     B_sim)
        self.alpha = nn.Parameter(torch.zeros(1))

    def _effective_B(self):
        if self.training and self.sigma_svd > 0:
            U, S, Vh = torch.linalg.svd(self.B_sim, full_matrices=False)
            S = S * (1.0 + self.sigma_svd * torch.randn_like(S))
            return U @ torch.diag(S) @ Vh
        return self.B_sim

    def forward(self, x):
        B_sz = x.size(0)
        C    = self.positions.shape[0]   # number of EEG channels
        pos  = self.positions
        if self.training and self.sigma_elec > 0:
            pos = pos + torch.randn_like(pos) * self.sigma_elec
        pos_b = pos.unsqueeze(0).expand(B_sz, -1, -1)

        with torch.no_grad():
            out  = self.reve(x, pos_b)
            feat = out.last_hidden_state if hasattr(out, 'last_hidden_state') else (out[0] if isinstance(out, (tuple, list)) else out)
            # Reshape to (B, C, d) — patch dim averaged, channel dim preserved.
            # This must happen INSIDE no_grad (REVE is frozen) but WITHOUT final
            # pooling so that B_sim channel-mixing can run outside no_grad and
            # receive a gradient through self.alpha.
            if feat.dim() == 4:                                     # (B, C, P, d)
                feat = feat.mean(dim=2)                             # → (B, C, d)
            elif feat.dim() == 3:                                   # (B, n_tok, d)
                n_tok = feat.shape[1]
                if n_tok % C == 0:
                    P = n_tok // C
                    feat = feat.reshape(B_sz, C, P, feat.shape[-1]).mean(dim=2)  # → (B, C, d)
                else:
                    # n_tok not divisible by C — can't group by channel; skip mixing
                    return feat.mean(dim=1)

        # B_sim channel-mixing — computed OUTSIDE no_grad so gradient reaches alpha
        if feat.dim() == 3 and feat.shape[1] == C:                 # (B, C, d)
            B_eff = self._effective_B()
            mixed = torch.einsum('ck,bkd->bcd', B_eff, feat)
            return (feat + self.alpha * mixed).mean(dim=1)          # (B, d)
        return feat                                                  # (B, d) fallback


class SeizureClassifier(nn.Module):
    def __init__(self, backbone, feat_dim, fusion_dim=128, freeze_backbone=True):
        super().__init__()
        self.backbone        = backbone
        self.freeze_backbone = freeze_backbone
        self.head = nn.Sequential(
            nn.Linear(feat_dim, fusion_dim), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(fusion_dim, fusion_dim), nn.ELU(), nn.Dropout(0.3),
            nn.Linear(fusion_dim, 2),
        )
    def forward(self, x):
        if self.freeze_backbone:
            with torch.no_grad():
                f = self.backbone(x)
        else:
            f = self.backbone(x)
        return self.head(f)


# Sanity check — backbones are instantiated fresh each LOPO fold (see training cell)
_bb_test = LeadfieldREVEExtractor(reve_model, POSITIONS, B_sim, 0.0, 0.0).to(DEVICE)
with torch.no_grad():
    _x = torch.randn(2, N_CH, T_WIN).to(DEVICE)
    print('Backbone output shape:', _bb_test(_x).shape)
del _x, _bb_test


## 7. Dataset & training

In [ ]:
class SeizureDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    def __len__(self):   return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]


def zscore(X):
    X = X.copy()
    X -= X.mean(axis=-1, keepdims=True)
    X /= (X.std(axis=-1, keepdims=True) + 1e-8)
    return X


def train_epoch(model, loader, opt, device):
    model.train()
    if model.freeze_backbone: model.backbone.eval()
    losses, preds, labels = [], [], []
    for Xb, yb in loader:
        Xb, yb = Xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(Xb)
        loss   = F.cross_entropy(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step()
        losses.append(loss.item())
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(yb.cpu().numpy())
    return np.mean(losses), balanced_accuracy_score(labels, preds)


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    preds, labels, probs = [], [], []
    for Xb, yb in loader:
        Xb = Xb.to(device)
        logits = model(Xb)
        probs.append(F.softmax(logits, -1).cpu().numpy())
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(yb.numpy())
    labels = np.array(labels); preds = np.array(preds)
    probs  = np.concatenate(probs, axis=0)
    bal_acc  = balanced_accuracy_score(labels, preds)
    auc      = roc_auc_score(labels, probs[:, 1]) if len(np.unique(labels)) > 1 else 0.5
    sens     = ((preds==1)&(labels==1)).sum() / max((labels==1).sum(), 1)
    spec     = ((preds==0)&(labels==0)).sum() / max((labels==0).sum(), 1)
    return {'bal_acc': bal_acc, 'auc': auc, 'sens': sens, 'spec': spec}


## 8. Leave-One-Patient-Out cross-validation

In [ ]:
BATCH_SIZE  = 32
MAX_EPOCHS  = 60
LR          = 1e-3
WEIGHT_DECAY= 1e-4
PATIENCE    = 12

all_results = {cond: {} for cond in ['A_baseline', 'B_leadfield', 'C_leadfield_aug']}

print('='*65)
print('REVE + B_sim  |  CHB-MIT  |  Leave-One-Patient-Out')
print('Conditions: A (baseline)  |  B (+B_sim)  |  C (B + jitter)')
print('='*65)

for test_pid in PATIENTS:
    print(f'\n--- Test patient: {test_pid} ---')

    # Re-instantiate backbones each fold so alpha is not accumulated across
    # patients (Bug 5 fix). The heavy frozen reve_model is shared — only the
    # single-scalar alpha is freshly initialised.
    backbone_A = LeadfieldREVEExtractor(reve_model, POSITIONS, B_sim, 0.0,   0.0  ).to(DEVICE)
    backbone_B = LeadfieldREVEExtractor(reve_model, POSITIONS, B_sim, 0.0,   0.0  ).to(DEVICE)
    backbone_C = LeadfieldREVEExtractor(reve_model, POSITIONS, B_sim, 0.005, 0.10 ).to(DEVICE)
    CONDITION_BACKBONES = {
        'A_baseline':      (backbone_A, True),
        'B_leadfield':     (backbone_B, False),
        'C_leadfield_aug': (backbone_C, False),
    }

    # Split
    X_test = zscore(patient_data[test_pid]['X'])
    y_test = patient_data[test_pid]['y']

    X_train_parts, y_train_parts = [], []
    for pid in PATIENTS:
        if pid == test_pid: continue
        X_train_parts.append(zscore(patient_data[pid]['X']))
        y_train_parts.append(patient_data[pid]['y'])
    X_train = np.concatenate(X_train_parts)
    y_train = np.concatenate(y_train_parts)

    # Balance training set (undersample majority)
    ictal_idx    = np.where(y_train == 1)[0]
    interict_idx = np.where(y_train == 0)[0]
    n_min = min(len(ictal_idx), len(interict_idx))
    rng   = np.random.default_rng(SEED)
    sel   = np.concatenate([
        rng.choice(ictal_idx,    n_min, replace=False),
        rng.choice(interict_idx, n_min, replace=False),
    ])
    rng.shuffle(sel)
    X_train, y_train = X_train[sel], y_train[sel]

    train_ds = SeizureDataset(X_train, y_train)
    test_ds  = SeizureDataset(X_test,  y_test)
    train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True)
    test_dl  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False)
    print(f'  train {len(train_ds)} (balanced)  |  test {len(test_ds)} '
          f'(ictal={y_test.sum()}, interictal={(y_test==0).sum()})')

    for cond_name, (bb, freeze_bb) in CONDITION_BACKBONES.items():
        model = SeizureClassifier(bb, REVE_FEAT_DIM, freeze_backbone=freeze_bb).to(DEVICE)
        opt   = torch.optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=LR, weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, MAX_EPOCHS)

        best_metrics = {'auc': 0}
        patience_ctr = 0
        for epoch in range(MAX_EPOCHS):
            train_loss, train_acc = train_epoch(model, train_dl, opt, DEVICE)
            sched.step()
            if (epoch+1) % 5 == 0 or epoch == 0:
                m = evaluate(model, test_dl, DEVICE)
                if m['auc'] > best_metrics['auc']:
                    best_metrics = m; patience_ctr = 0
                else:
                    patience_ctr += 5
                if patience_ctr >= PATIENCE:
                    print(f'    [{cond_name}] early stop ep{epoch+1} | '
                          f'best AUC={best_metrics["auc"]:.3f}  sens={best_metrics["sens"]:.3f}')
                    break
        all_results[cond_name][test_pid] = best_metrics
        print(f'  {cond_name:20s}  AUC={best_metrics["auc"]:.3f}  '
              f'sens={best_metrics["sens"]:.3f}  spec={best_metrics["spec"]:.3f}')
        del model, opt, sched
        torch.cuda.empty_cache()


## 9. Results

In [ ]:
print('\n' + '='*70)
print('CHB-MIT LOPO Results  —  Frozen REVE + B_sim')
print('='*70)
print(f'{"Patient":>10}  {"A AUC":>8}  {"B AUC":>8}  {"C AUC":>8}  {"B-A":>8}')
print('-'*50)
for pid in PATIENTS:
    a = all_results['A_baseline'][pid]['auc']
    b = all_results['B_leadfield'][pid]['auc']
    c = all_results['C_leadfield_aug'][pid]['auc']
    print(f'{pid:>10}  {a:>8.3f}  {b:>8.3f}  {c:>8.3f}  {b-a:>+8.3f}')
print('-'*50)
for metric_name, metric_key in [('Mean AUC', 'auc'), ('Sens', 'sens'), ('Spec', 'spec')]:
    row = f'{metric_name:>10}'
    for cond in ['A_baseline','B_leadfield','C_leadfield_aug']:
        vals = [all_results[cond][p][metric_key] for p in PATIENTS]
        row += f'  {np.mean(vals):>8.3f}'
    print(row)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('CHB-MIT — Frozen REVE + B_sim  |  Cross-Patient LOPO', fontsize=13)

conds  = ['A_baseline', 'B_leadfield', 'C_leadfield_aug']
labels = ['A — Baseline', 'B — +B_sim', 'C — B+jitter']
colors = ['#85B7EB', '#5DCAA5', '#AFA9EC']
x = np.arange(len(PATIENTS)); w = 0.25

ax = axes[0]
for i,(cond,label,col) in enumerate(zip(conds,labels,colors)):
    vals = [all_results[cond][p]['auc'] for p in PATIENTS]
    ax.bar(x+i*w, vals, w, label=label, color=col, edgecolor='white')
ax.set_xticks(x+w); ax.set_xticklabels(PATIENTS)
ax.axhline(0.5, color='red', lw=1, ls='--', label='Chance')
ax.set_ylabel('AUC'); ax.set_title('Per-Patient AUC')
ax.set_ylim(0,1.05); ax.legend(fontsize=8); ax.grid(alpha=0.3,axis='y')

ax = axes[1]
base_aucs = np.array([all_results['A_baseline'][p]['auc'] for p in PATIENTS])
b_aucs    = np.array([all_results['B_leadfield'][p]['auc'] for p in PATIENTS])
c_aucs    = np.array([all_results['C_leadfield_aug'][p]['auc'] for p in PATIENTS])
ax.scatter(base_aucs, b_aucs-base_aucs, s=80, c='#5DCAA5', label='+B_sim', zorder=3, edgecolors='white')
ax.scatter(base_aucs, c_aucs-base_aucs, s=80, c='#AFA9EC', label='+B_sim+jitter', zorder=3, edgecolors='white')
ax.axhline(0, color='gray', ls='--', lw=0.8)
ax.set_xlabel('Baseline AUC'); ax.set_ylabel('ΔAUC vs baseline')
ax.set_title('Who benefits?'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('exp004_seizure_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
out = {}
for cond in conds:
    out[cond] = {pid: {k: float(v) for k,v in m.items()}
                 for pid, m in all_results[cond].items()}

with open('exp004_results.json', 'w') as f:
    json.dump({'experiment': 'EXP_004', 'backbone': 'REVE-Base',
               'dataset': 'CHB-MIT', 'evaluation': 'LOPO',
               'patients': PATIENTS, 'results': out}, f, indent=2)
print('Saved exp004_results.json')
